In [ ]:
# To pull User data from Replay files, and then pull User data from PS.com

# Getting user data
------------

In [13]:
# Some basic imports
import numpy as np
import pandas as pd

import json, re, time
import logging, requests
import itertools

from pathlib import Path
from typing import Optional
from dataclasses import dataclass, asdict
from urllib.parse import urlencode, urljoin

# Useful 'constants'
REPLAY_DIR = Path('./replays/gen9-randombattle')
USER_DIR = Path('./users')

SEARCH_BASE_URL = "https://replay.pokemonshowdown.com/api/replays/search"
REPLAY_BASE_URL = "https://replay.pokemonshowdown.com/"
USER_BASE_URL = "https://pokemonshowdown.com/users/"

logger = logging.getLogger()
logger.setLevel(logging.ERROR)

@dataclass 
class User:
    username: str
    userid: str
    registertime: int
    group: int
    ratings: dict

In [5]:
# =====================================
# replays |-> users_raw.txt
# =====================================
user_list = []

for item in REPLAY_DIR.iterdir():
    if (item.is_file() and item.name[0]!='.'): # skipping hidden files
        try:
            with open(REPLAY_DIR/f'{item.name}', 'r') as file:
                x = json.load(file)
                for user in x.get('players',[]):
                    user_list.append(user)
        except UnicodeDecodeError as e:
            print(f"Error with {item.name}")
    
    elif (item.name[0]=='.') :
        continue
    else: 
        print(f'Could not open {item.name}')
        continue

# can change to 'a' mode later
with open('./users_raw.txt', 'w') as file: 
    for user in user_list : file.write(user + '\n')

In [6]:
# ===========================
def get_user(user_id: str):
    '''
    Return a `User` object pulled from PS using `user_id`, 'cleaning' the string first.
    '''

    # clean by deleting anything NOT in {A-Z,a-z,0-9}, and then |-> lowercase
    cleaner = lambda s : re.sub("[^A-Za-z0-9]", "", s).lower()
    user_id = cleaner(user_id)

    url = urljoin(USER_BASE_URL, f'{user_id}.json')
    
    response = requests.get(url, timeout=10)
    
    try:
        data = response.json()
    except json.JSONDecodeError as e:
        logger.error(f"failed to parse user {user_id}: {e}")
        return None

    # ===========================
    # 'Unregistered' users are treated much like nonexistant users, but the former can 
    # have ratings. Hence, we suppress `404` errors and instead check whether a 
    # User's `ratings` field is empty or not.
    if data.get("ratings", {}) == {} :
        return None
    
    else :
        return User(
            username = data.get("username", ""),
            userid = data.get("userid", ""),
            registertime = data.get("registertime", 0),
            group = data.get("group", 0),
            ratings = data.get("ratings", dict())
        )

In [26]:
users_raw = list(set(users_raw)) # remove duplicates

In [28]:
# =====================================
# users_raw |-> PokemonShowdown |-> `./users/<file.json>`
# =====================================

# grab names from file into array 
with open('users_raw.txt', 'r') as file:
    users_raw = file.readlines()
users_raw = list(map(lambda s : s[:-1], users_raw)) # strip tailing '\n' from each entry

user_errors = [] # anyone we fail(ed) to get data for 

i = 7500
for username in users_raw[7500:]:
    if (i%20 == 0) : print(f"Now working on user #{i}.")
    # -----------------------
    # `get_user()`
    user = get_user(username)
    time.sleep(0.3)
    i += 1
    # -----------------------
    
    if user is not None :
        with open(USER_DIR/f'{user.userid}.json', 'w') as file:
            json.dump(asdict(user), file)
    else : user_errors.append(username)

if user_errors == []:
    print(f"Successfully pulled data for all {len(users_raw)} users.")
else : 
    print(f"Failed to get data for {len(user_errors)} users; writing to file.")
    with open('user_errors.txt', 'a') as file:
        for user in user_errors:
            file.write(user + '\n')

Now working on user #7500.
Now working on user #7520.
Now working on user #7540.
Now working on user #7560.
Now working on user #7580.
Now working on user #7600.
Now working on user #7620.
Now working on user #7640.
Now working on user #7660.
Now working on user #7680.
Now working on user #7700.
Now working on user #7720.
Now working on user #7740.
Now working on user #7760.
Now working on user #7780.
Now working on user #7800.
Now working on user #7820.
Now working on user #7840.
Now working on user #7860.
Now working on user #7880.
Now working on user #7900.
Now working on user #7920.
Now working on user #7940.
Now working on user #7960.
Now working on user #7980.
Now working on user #8000.
Now working on user #8020.
Now working on user #8040.
Now working on user #8060.
Now working on user #8080.
Now working on user #8100.
Now working on user #8120.
Now working on user #8140.
Now working on user #8160.
Now working on user #8180.
Now working on user #8200.
Now working on user #8220.
N

ReadTimeout: HTTPSConnectionPool(host='pokemonshowdown.com', port=443): Read timed out. (read timeout=10)